<b>Quantization is the process of reducing the precision of a model's weights and activations (typically from FP32 to INT8 or FP16). This significantly reduces model size and improves inference speed, especially on CPUs, with minimal loss in accuracy.</b>

<p>There are two primary ways to quantize BERT: Dynamic Quantization (easiest for CPUs) and Load-time Quantization (using bitsandbytes for GPUs).</p>

<b>1. PyTorch Dynamic Quantization (CPU)</b>

<p>Dynamic quantization is the most straightforward method. It quantizes the weights to int8 ahead of time, while the activations are quantized dynamically during inference. This is ideal for BERT models deployed on servers or edge devices with CPUs.</p>

In [2]:
import os
import torch
from transformers import BertForSequenceClassification, BertTokenizer

# 1. Load your fine-tuned BERT model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# 2. Apply Dynamic Quantization
# We target the 'Linear' layers, as they take up most of BERT's parameters
quantized_model = torch.quantization.quantize_dynamic(
    model, 
    {torch.nn.Linear}, 
    dtype=torch.qint8
)

# 3. Compare sizes
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    print(f"Size (MB): {os.path.getsize('temp.p')/1e6:.2f}")
    os.remove('temp.p')

print("Original BERT size:")
print_size_of_model(model)
print("Quantized BERT size:")
print_size_of_model(quantized_model)

/home/jovyan/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from t

Original BERT size:
Size (MB): 438.01
Quantized BERT size:
Size (MB): 181.48


<b>2. Hugging Face 4-bit/8-bit Quantization (GPU)</b>

<p>If you are running BERT on a GPU and want to save VRAM, you can use the bitsandbytes integration. This is widely used in modern LLM pipelines but works perfectly for BERT as well.</p>

In [8]:
import torch
from transformers import BertForSequenceClassification, BitsAndBytesConfig

# 1. THE FIX: Manually add the missing attribute to the BERT class
BertForSequenceClassification._no_split_modules = ["BertLayer"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 2. Now device_map="auto" or "cuda:0" will work
model_4bit = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Model loaded. Memory: {model_4bit.get_memory_footprint() / 1e6:.2f} MB")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


NotImplementedError: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.

<p>For BERT, the most reliable way to load in 4-bit without hitting the "meta tensor" error is to let transformers handle the device placement internally by just specifying device_map="cuda:0" (or just 0) as a string, rather than a dictionary.</p>

<p>Hugging Face recently released quanto (now part of the optimum library). It is often better than bitsandbytes for encoder models like BERT because it doesn't rely on the complex device_map logic that is causing your errors.</p>

In [11]:
from transformers import BertForSequenceClassification, BertTokenizer
from optimum.quanto import quantize, freeze, qint8  # Import qint8 here
import torch

model_id = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
model = BertForSequenceClassification.from_pretrained(model_id).to("cuda")

# 1. Use qint8 instead of torch.int8
quantize(model, weights=qint8)

# 2. Freeze the weights to finalize the quantization
freeze(model)

print(f"Quantized Model Memory: {model.get_memory_footprint() / 1e6:.2f} MB")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly i

Quantized Model Memory: 437.94 MB


<p>If you want to avoid the manual quantize and freeze steps, Hugging Face transformers now supports a QuantoConfig object. This is the cleanest way to handle BERT quantization because it handles the weight conversion automatically during the loading process.</p>

In [12]:
from transformers import BertForSequenceClassification, QuantoConfig
import torch

# Define the quantization config
# Options for weights: "int8", "float8", "int4", "int2"
quant_config = QuantoConfig(weights="int8")

# Load the model with the config
# For BERT, we still omit device_map="auto" to avoid the previous error
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    quantization_config=quant_config
).to("cuda")

print(f"Model successfully loaded and quantized.")
print(f"Memory Footprint: {model.get_memory_footprint() / 1e6:.2f} MB")

ImportError: cannot import name 'QuantoConfig' from 'transformers' (/home/jovyan/.venv/lib/python3.11/site-packages/transformers/__init__.py)

In [13]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer
from optimum.quanto import quantize, freeze, qint8

model_id = "bert-base-uncased"

# 1. Load the model to CPU first to avoid the 'meta tensor' error
# Then move to GPU once initialized
model = BertForSequenceClassification.from_pretrained(model_id)
model.to("cuda")

# 2. Quantize the weights using the correct quanto type (qint8)
# This replaces the Linear layers with QLinear layers
quantize(model, weights=qint8)

# 3. Freeze the model
# This converts the float weights into the actual quantized int8 format
freeze(model)

print(f"Model successfully quantized to INT8.")
print(f"Memory Footprint: {model.get_memory_footprint() / 1e6:.2f} MB")

# 4. Simple Inference Test
tokenizer = BertTokenizer.from_pretrained(model_id)
inputs = tokenizer("Quantization is working!", return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model(**inputs)
    print("Inference successful. Logic check:", outputs.logits.shape)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly i

Model successfully quantized to INT8.
Memory Footprint: 437.94 MB
Inference successful. Logic check: torch.Size([1, 2])
